# 2장 2강: Likelihood와 Maximum Likelihood — 실습문제

## 실습 목표

- 조건을 만족하는 고객만 골라 구매 비율을 구합니다.
- 실제 구매 여부에 맞는 확률을 선택하고, 그 확률들을 곱해 우도(Likelihood)를 구합니다.
- 몇 가지 구매 확률 후보를 비교하여 최대우도추정(MLE)의 뜻을 익힙니다.
- 로그우도와 음의 로그우도(NLL)를 단계별로 계산하고 합계와 평균을 구분합니다.

## 실습 환경 / 데이터

Python과 NumPy를 사용합니다. 외부 데이터 파일은 필요하지 않습니다. 모든 데이터와 수학적 계산 방법은 제공된 「2장 2강: Likelihood와 Maximum Likelihood」의 본문과 연습문제에서 가져왔습니다. **0은 미구매, 1은 구매**입니다. 이 자료에서는 확률이 주어졌을 때의 계산을 연습합니다.

## 실습 구성과 교안 대응

| 구분 | 실습 내용 | 교안 근거 |
|---|---|---|
| 필수 1 / 문제 1-1 | 전체·장바구니 추가 여부별 구매 비율 | 1절 본문: 고객 10명의 조건부 확률 예제 |
| 필수 1 / 문제 1-2 | 공통 구매 확률 0.6과 0.8의 우도 | 2절 본문: 관측값 [1,0,1,1,0] 비교 |
| 필수 2 / 문제 2-1 | 네 후보의 우도와 구매 비율로 MLE 확인 | 3절 본문: 후보 0.2, 0.4, 0.6, 0.8 |
| 필수 2 / 문제 2-2 | 고객별 NLL, 전체 로그우도, 합계·평균 NLL | 교안 후반부: y=[1,0,1], p=[0.8,0.3,0.7] 예제 |
| 과제 / 문제 3-1 | 세 후보의 우도와 MLE 복습 | 연습문제 3: y=[1,1,0,1], 후보 0.5, 0.75, 0.9 |
| 과제 / 문제 3-2 | 고객별 확률이 다른 두 모델의 우도 | 연습문제 2: 두 모델의 고객별 확률 비교 |

**교안 연습문제 전체를 그대로 옮긴 자료는 아닙니다.** 본문 예제와 연습문제의 일부를 필수 실습 4문제와 복습 과제 2문제로 재구성했습니다. 과제도 앞에서 연습한 식에 교안의 숫자를 넣어 계산하는 기초 수준입니다.

## 진행 안내

1. NumPy를 불러오는 셀부터 위에서 아래로 실행합니다.
2. 각 문제의 표와 기호 설명을 읽고 계산하세요. 함수가 제시된 경우 그대로 사용해도 됩니다.
3. 손으로 로그값을 구하거나 공식을 증명할 필요는 없습니다. 로그 계산은 교안의 `np.log()`를 사용합니다.
4. 계산 중간에는 반올림하지 않습니다. 마지막 출력에만 `round()` 또는 `np.round()`를 사용합니다.
5. 각 문제의 해석 질문 4개에 계산값과 짧은 설명을 적습니다.

## 먼저 읽는 용어와 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 관측 데이터 D / 실제값 y | 이미 확인한 실제 결과들 / 한 고객의 구매 여부. 코드에서는 y에 여러 결과를 순서대로 넣음 |
| 확률 p | 구매할 가능성을 나타내는 수. 0.6은 60% |
| 1−p | 구매하지 않을 확률. 구매와 미구매의 확률을 더하면 1 |
| 베르누이 분포 | 결과가 0과 1 두 가지인 경우의 확률을 나타내는 방법 |
| 조건부 확률 | 특정 조건을 만족하는 경우만 모아서 보는 확률 |
| P(구매 ∣ 장바구니 추가) | 장바구니에 추가했다는 조건에서의 구매 확률. 세로줄 뒤가 조건 |
| 파라미터 | 모델의 확률을 결정하는 값. 공통 구매 확률 예제에서는 p |
| 독립 가정 | 여기서는 p가 정해졌을 때 한 고객의 구매 여부가 다른 고객의 구매 여부에 영향을 주지 않는다고 가정함 |
| 우도 L(p) / Likelihood | 실제로 관측한 데이터를 고정하고, p에 따라 그 데이터에 부여되는 확률을 비교하는 값 |
| 최대우도추정 / MLE | 관측 데이터의 우도가 가장 커지는 파라미터를 찾는 것 |
| p̂ / ‘피 햇’ | 데이터를 보고 추정한 p. 위에 붙은 작은 모자는 추정값이라는 표시 |
| ln / 자연로그 | 밑이 자연상수 e인 로그. 코드의 `np.log()`로 계산 |
| 로그우도 / Log-Likelihood | 우도에 자연로그를 적용한 값. 관측 확률 각각의 로그를 더해 계산할 수 있음 |
| 음의 로그우도 / NLL | 로그우도의 부호를 바꾼 값. 이 자료에서는 작을수록 좋다고 보는 손실값 |
| Total / Mean | 합계 / 평균. 평균은 합계를 데이터 개수로 나눈 값 |
| ≈ / Σ / Π | 대략 같음 / 여러 값을 더함 / 여러 값을 곱함 |
| p³ / p ** 3 | 둘 다 p를 세 번 곱한다는 뜻 |
| 인덱스 | 배열의 위치 번호. 첫 번째 위치는 0, 두 번째는 1 |

**가장 먼저 구분할 것:** p는 항상 ‘구매 확률’입니다. 실제로 구매했다면 p를, 구매하지 않았다면 1−p를 사용합니다. 실제 결과에 해당하는 확률을 고른 다음 우도와 NLL을 계산합니다.

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 숫자 배열을 계산하는 도구이며, `np`는 NumPy를 짧게 부르는 이름입니다.

In [1]:
import numpy as np

---

## 필수 실습 1. 조건부 확률과 우도

### 문제 1-1. 조건에 따라 구매 비율 계산하기

#### 문제 설명

교안 1절 본문의 고객 10명입니다. 각 배열의 같은 위치는 같은 고객을 뜻합니다.

```python
add_cart = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0])
purchase = np.array([1, 1, 1, 1, 0, 0, 1, 0, 0, 0])
```

`add_cart`의 1은 장바구니 추가, 0은 추가하지 않음입니다. `purchase`의 1은 구매, 0은 미구매입니다.

| 구분 | 구매 | 미구매 | 합계 |
|---|---:|---:|---:|
| 장바구니 추가 | 4 | 2 | 6 |
| 장바구니 추가 안 함 | 1 | 3 | 4 |
| 전체 | 5 | 5 | 10 |

#### 요구사항

1. 주어진 두 배열을 만들고 `np.mean(purchase)`로 전체 구매 비율을 출력하세요.
2. `purchase[add_cart == 1]`과 `purchase[add_cart == 0]`을 각각 출력하세요.
3. 두 배열 각각에 `np.mean()`을 적용해 조건별 구매 비율을 출력하세요. 비율은 소수점 아래 네 자리로 표시합니다.
4. 아래 Q1~Q4에 답하세요.

#### 작성 도움말

`purchase[add_cart == 1]`은 장바구니에 추가한 고객의 구매 결과만 고릅니다. 0·1 배열의 평균은 ‘1의 개수 ÷ 배열의 길이’입니다. 조건부 확률은 **조건을 만족하는 고객을 먼저 고른 다음**, 그 안에서 구매 비율을 구합니다.

#### 해석 질문

**Q1.** 전체 구매 비율은 얼마이며, 어떤 수를 어떤 수로 나누나요?

**Q2.** 장바구니에 추가한 고객의 구매 결과 배열과 구매 비율은 무엇인가요?

**Q3.** 장바구니에 추가하지 않은 고객의 구매 결과 배열과 구매 비율은 무엇인가요?

**Q4.** P(구매 ∣ 장바구니 추가)를 구할 때 4÷10을 쓰면 왜 다른 계산이 되나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [2]:
# 문제 1-1. 조건에 따라 구매 비율 계산하기
# 1. 문제에 주어진 배열과 값을 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 계산 결과의 뜻을 적으세요.


add_cart = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0])
purchase = np.array([1, 1, 1, 1, 0, 0, 1, 0, 0, 0])
# 첫 번째 고객은 add_cart가 1이고 purchase도 1이니까 장바구니 추가 후 구매도 완료
# 다섯 번째 고객은 add_cart가 1이고 purchase가 0이니까 장바구니 추가 후 미구매


purchase_rate = np.mean(purchase)
cart_purchase = purchase[add_cart==1]
no_cart_purchase = purchase[add_cart==0]

rate_cart = np.mean(cart_purchase)
rate_no_cart = np.mean(no_cart_purchase)

print("전체 구매 비율:", round(purchase_rate, 4))
print("장바구니 추가 고객의 구매 결과:", cart_purchase)
print("추가하지 않은 고객의 구매 결과:", no_cart_purchase)
print("추가 고객의 구매 비율:", round(rate_cart, 4))
print("추가하지 않은 고객의 구매 비율:", round(rate_no_cart, 4))

전체 구매 비율: 0.5
장바구니 추가 고객의 구매 결과: [1 1 1 1 0 0]
추가하지 않은 고객의 구매 결과: [1 0 0 0]
추가 고객의 구매 비율: 0.6667
추가하지 않은 고객의 구매 비율: 0.25


#### 답변 작성란

**Q1.** 전체 구매 비율은 얼마이며, 어떤 수를 어떤 수로 나누나요?

답변: 
- 구매한 5명을 전체 10명으로 나눈다.
- 5 / 10 = 0.5, 50%
- 미구매 고객도 전체 고객수에 포함된다.

**Q2.** 장바구니에 추가한 고객의 구매 결과 배열과 구매 비율은 무엇인가요?

답변: 
- 배열은 [1, 1, 1, 1, 0, 0]
- 이 안에서 실제 구매한 고객은 4명이고, 고객 수가 6명이기 때문에
- 4 / 6 = 약 67% 정도
- 조건을 만족한 6명만을 분모로 사용한다.

**Q3.** 장바구니에 추가하지 않은 고객의 구매 결과 배열과 구매 비율은 무엇인가요?

답변:
- 배열은 [1, 0, 0, 0]이고, 구매 1명을 해당 그룹의 4명으로 나누면
- 1/4 = 0.25, 즉 25%

**Q4.** P(구매 ∣ 장바구니 추가)를 구할 때 4÷10을 쓰면 왜 다른 계산이 되나요?

답변:
- 4/10은 전체 10명 중 장바구니에 추가하고 구매도 한 4명의 비율
- 질문은 추가한 고객 안에서의 구매 비율이므로 분모를 6으로 바꾸어 4/6으로 계산한다.

### 문제 1-2. 실제 관측 결과의 확률을 곱해 Likelihood 비교하기

#### 문제 설명

교안 2절 본문의 실제 구매 결과는 `y = [1, 0, 1, 1, 0]`입니다. 공통 구매 확률을 모델 A는 p=0.6, 모델 B는 p=0.8로 둡니다. 각 모델에서 p가 정해졌을 때 고객들의 구매 여부는 서로 독립이라고 가정합니다.

실제 구매 고객(y=1)에는 p, 실제 미구매 고객(y=0)에는 1−p를 사용합니다. 이렇게 고른 확률을 모두 곱한 값이 이 관측 데이터의 우도입니다.

#### 요구사항

1. y 배열과 두 모델의 공통 구매 확률을 만드세요.
2. 아래 교안의 식을 두 모델에 각각 적용하여 실제 결과에 해당하는 확률 배열을 출력하세요.
3. `np.prod()`로 두 우도를 출력하고 큰 값을 선택하세요. 우도는 소수점 아래 다섯 자리로 표시합니다.
4. Q1~Q4에 답하세요.

#### 수식과 작성 도움말

```python
observed_prob = (p ** y) * ((1 - p) ** (1 - y))
```

수식으로는 `p^y × (1−p)^(1−y)`입니다. `**`는 거듭제곱입니다. 이번처럼 0<p<1일 때 어떤 양수의 0제곱은 1, 1제곱은 자기 자신입니다.

- y=1이면 p¹×(1−p)⁰=p×1=p입니다.
- y=0이면 p⁰×(1−p)¹=1×(1−p)=1−p입니다.

이 식은 구매와 미구매에 맞는 확률을 하나의 코드로 선택합니다.

#### 해석 질문

**Q1.** 두 모델에서 실제 결과에 해당하는 확률 배열은 각각 무엇인가요?

**Q2.** 두 모델의 우도는 얼마이며, 이 데이터에서는 어느 모델을 선택하나요?

**Q3.** 구매 확률 0.8이 0.6보다 큰데도 모델 B의 우도가 작을 수 있는 이유는 무엇인가요?

**Q4.** Probability와 Likelihood에서는 각각 무엇을 고정하나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [3]:
# 문제 1-2. 실제 관측 결과의 확률을 곱해 Likelihood 비교하기
# 1. 문제에 주어진 배열과 값을 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 계산 결과의 뜻을 적으세요.


y = np.array([1, 0, 1, 1, 0])
p_a = 0.6
p_b = 0.8

# 바뀌지 않는 것 = y  / 바뀌는것 p(확률) 예제에서는 p_a와 p_b
observed_a = (p_a ** y) * ((1-p_a) ** (1-y)) # 우도 계산
observed_b = (p_b ** y) * ((1-p_b) ** (1-y)) # 우도 계산

likelihood_a = np.prod(observed_a)
likelihood_b = np.prod(observed_b)

print('모델 A 실제 결과의 확률 :', np.round(observed_a, 2))
print('모델 B 실제 결과의 확률 :', np.round(observed_b, 2))

print('\n모델 A Likelihood :', np.round(likelihood_a, 2))
print('모델 B Likelihood :', np.round(likelihood_b, 2))

모델 A 실제 결과의 확률 : [0.6 0.4 0.6 0.6 0.4]
모델 B 실제 결과의 확률 : [0.8 0.2 0.8 0.8 0.2]

모델 A Likelihood : 0.03
모델 B Likelihood : 0.02


#### 답변 작성란

**Q1.** 두 모델에서 실제 결과에 해당하는 확률 배열은 각각 무엇인가요?

답변: 
- 모델 A는 [0.6, 0.4, 0.6, 0.6, 0.4]
- 모델 B는 [0.8, 0.2, 0.8, 0.8, 0.2]

**Q2.** 두 모델의 우도는 얼마이며, 이 데이터에서는 어느 모델을 선택하나요?

답변: 
- A는 0.3456
- B는 0.02048
- 실제 같은 데이터에 부여한 전체 확률인 우도가 더 큰 A를 선택할 수 있다.

**Q3.** 구매 확률 0.8이 0.6보다 큰데도 모델 B의 우도가 작을 수 있는 이유는 무엇인가요?

답변: 
- 실제 결과에는 구매 3건뿐 아니라 미구매 2건도 포함되어 있음
- B는 구매에는 0.8을 주지만 미구매에는 0.2를 준다.
- 모든 결과를 반영하여 곱해보면 0.8³ x 0.2³ = 0.2048
- A 0.6³ x 0.4³ = 0.03456보다 작을 수 있다. 

**Q4.** Probability와 Likelihood에서는 각각 무엇을 고정하나요?

답변:
- Probability는 확률 P와 같은 모델 파라미터를 정해놓고 결과의 확률을 본다.
- Likelihood는 이미 관측한 결과를 고정하고 P를 바꾸어 비교

---

## 필수 실습 2. MLE와 NLL

### 문제 2-1. 네 가지 구매 확률 후보에서 MLE 확인하기

#### 문제 설명

교안 3절의 `y = [1, 0, 1, 1, 0]`과 후보 `p = [0.2, 0.4, 0.6, 0.8]`을 사용합니다. 모든 고객이 같은 구매 확률 p를 가지며, p가 주어졌을 때 구매 여부가 서로 독립인 베르누이 모델을 가정합니다.

MLE는 관측 데이터의 우도를 가장 크게 만드는 파라미터를 찾는 것입니다. 이 공통 p 모델에서는 교안의 식 `p_mle = np.mean(y)`로 MLE를 구할 수 있습니다.

#### 요구사항

1. y와 후보 배열을 만드세요. 아래 우도 함수는 그대로 사용해도 됩니다.
2. 후보별 우도를 구해 소수점 아래 다섯 자리로 출력하세요.
3. `np.argmax()`로 최대 우도의 위치 번호를 구하고, 후보 배열에서 해당 p를 꺼내 출력하세요.
4. `np.mean(y)`로 MLE를 구해 출력한 뒤 Q1~Q4에 답하세요.

#### 제공 코드와 읽는 방법

```python
def likelihood(p, y):
    observed_prob = (p ** y) * ((1 - p) ** (1 - y))
    return np.prod(observed_prob)

values = np.array([likelihood(p, y) for p in candidates])
best_index = np.argmax(values)
best_p = candidates[best_index]
```

`candidates`는 후보를 모아 둔 배열입니다. `values`에는 같은 순서로 계산한 우도가 들어갑니다. `np.argmax(values)`는 p가 아니라 **가장 큰 우도의 위치 번호**를 반환합니다. Python의 위치 번호는 0부터 시작합니다.

#### 해석 질문

**Q1.** 관측 데이터의 구매·미구매 수는 각각 몇 명이며 L(p)는 어떤 식인가요?

**Q2.** 후보별 우도와 가장 좋은 후보 p는 무엇인가요?

**Q3.** np.argmax(values)의 결과와 선택한 p가 왜 서로 다른가요?

**Q4.** 관측 평균으로 구한 MLE는 얼마이며, 어떤 모델 가정에서 이 식을 쓰나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [4]:
# 문제 2-1. 네 가지 구매 확률 후보에서 MLE 확인하기
# 1. 문제에 주어진 배열과 값을 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 계산 결과의 뜻을 적으세요.

# 1
y = np.array([1, 0, 1, 1, 0])
candidates = np.array([0.2, 0.4, 0.6, 0.8])

def likelihood(p, y):
    observed_prob = (p ** y) * ((1 - p) ** (1 - y))
    return np.prod(observed_prob)

values = np.array([likelihood(p, y) for p in candidates])
best_index = np.argmax(values)
best_p = candidates[best_index]

p_mle = np.mean(y)

print('구매확률 후보 :', candidates)
print("후보별 Likelihood:", np.round(values, 5))
print("최대 우도의 위치 번호:", best_index)
print("후보 중 가장 좋은 p:", best_p)
print("구매 비율로 구한 MLE:", p_mle)

구매확률 후보 : [0.2 0.4 0.6 0.8]
후보별 Likelihood: [0.00512 0.02304 0.03456 0.02048]
최대 우도의 위치 번호: 2
후보 중 가장 좋은 p: 0.6
구매 비율로 구한 MLE: 0.6


#### 답변 작성란

**Q1.** 관측 데이터의 구매·미구매 수는 각각 몇 명이며 L(p)는 어떤 식인가요?

답변: 
- 구매 3명, 미구매 2명
- 구매 확률 P를 세 번, 미구매 확률 1-P를 두 번 곱한다.
- L(p) = p³(1-p)²

**Q2.** 후보별 우도와 가장 좋은 후보 p는 무엇인가요?

답변: 
- p=0.2, 0.4, 0.6, 0.8 순서대로 우도는 0.005, 0.023, 0.034, 0.020으로
- 0.034가 가장 크기 때문에 해당 후보 p=0.6을 선택한다.

**Q3.** np.argmax(values)의 결과와 선택한 p가 왜 서로 다른가요?

답변: 
- np.argmax(values)는 가장 큰 값의 위치 번호 2를 반환
- candidates[2]로 그 위치를 정확하게 보면 구매확률 0.6을 얻을 수 있다.

**Q4.** 관측 평균으로 구한 MLE는 얼마이며, 어떤 모델 가정에서 이 식을 쓰나요?

답변:
- 합계 3을 고객 수 5로 나누어 0.6, 모든 고객이 같은 구매 확률 p를 사용하고
- p가 주어졌을 때 구매여부가 서로 독립인 베르누이 모델에서 이 식을 사용한다.

### 문제 2-2. 고객별 NLL과 합계·평균 계산하기

#### 문제 설명

교안 후반부의 세 고객 예제입니다. 고객마다 구매 확률 p가 다릅니다. 입력 특성과 모델의 파라미터가 주어졌을 때 구매 여부가 서로 조건부 독립이라고 가정합니다.

| 고객 | 실제 구매 여부 y | 구매 확률 p |
|---|---:|---:|
| A | 1 | 0.8 |
| B | 0 | 0.3 |
| C | 1 | 0.7 |

`ln`은 자연로그이고, 코드에서는 `np.log()`로 계산합니다. 손으로 로그값을 계산하거나 로그 공식을 증명할 필요는 없습니다.

#### 요구사항

1. y와 p 배열을 만든 뒤 실제 결과의 확률 배열 `observed_prob`를 구해 출력하세요.
2. 아래 교안의 NLL 식으로 고객별 NLL을 구하세요.
3. 고객별 NLL의 합계는 `np.sum()`, 평균은 `np.mean()`으로 구하세요.
4. `np.sum(np.log(observed_prob))`로 전체 로그우도를 구하세요. 로그우도와 NLL은 소수점 아래 네 자리로 출력하고 Q1~Q4에 답하세요.

#### 수식과 작성 도움말

```python
nll_each = -(y * np.log(p) + (1 - y) * np.log(1 - p))
```

이 식은 `−[y×ln(p)+(1−y)×ln(1−p)]`입니다. 대괄호 안을 모두 계산한 다음 앞의 마이너스로 부호를 바꿉니다.

- 구매 고객(y=1): `−ln(p)`가 남습니다.
- 미구매 고객(y=0): `−ln(1−p)`가 남습니다.

Total NLL은 고객별 NLL의 합, Mean NLL은 그 합을 고객 수 3으로 나눈 값입니다.

#### 해석 질문

**Q1.** 고객별 실제 결과의 확률은 무엇이며, B에게 0.3을 그대로 쓰지 않는 이유는 무엇인가요?

**Q2.** 고객별 NLL은 각각 얼마이며, B와 C의 NLL이 같은 이유는 무엇인가요?

**Q3.** Total NLL과 Mean NLL은 각각 얼마이며, 평균은 무엇으로 나누나요?

**Q4.** 전체 로그우도는 얼마이며, Total NLL과 어떤 관계인가요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [5]:
# 문제 2-2. 고객별 NLL과 합계·평균 계산하기
# 1. 문제에 주어진 배열과 값을 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 계산 결과의 뜻을 적으세요.

y = np.array([1, 0, 1])
p = np.array([0.8, 0.3, 0.7])


# NLL을 작게 만드는 것은 '실제 결과의 확률'을 크게 만든다.
# 로그 우도는 큰쪽, NLL은 작은 쪽 결과를 고른다.
observed_prob = (p ** y) * ((1-p) ** (1-y))
nll_each = -(y * np.log(p) + (1 - y) * np.log(1 - p))

total_nll = np.sum(nll_each)
mean_nll = np.mean(nll_each)
log_likelihood = np.sum(np.log(observed_a))

print('실제 결과의 확률 :', np.round(observed_prob, 4))
print("고객별 NLL:", np.round(nll_each, 4))
print("Total NLL:", round(total_nll, 4))
print("Mean NLL:", round(mean_nll, 4))
print("전체 Log-Likelihood:", round(log_likelihood, 4))

실제 결과의 확률 : [0.8 0.7 0.7]
고객별 NLL: [0.2231 0.3567 0.3567]
Total NLL: 0.9365
Mean NLL: 0.3122
전체 Log-Likelihood: -3.3651


#### 답변 작성란

**Q1.** 고객별 실제 결과의 확률은 무엇이며, B에게 0.3을 그대로 쓰지 않는 이유는 무엇인가요?

답변: 
- [0.8, 0.7, 0.7]이다.
- B의 실제 결과는 미구매이기 때문에
- 주어진 0.3은 구매 확률이므로 실제 미구매 결과의 확률은 1-0.3 = 0.7이다.

**Q2.** 고객별 NLL은 각각 얼마이며, B와 C의 NLL이 같은 이유는 무엇인가요?

답변: 
- A는 -ln(0.8) = 약 0.223, B와 C는 모두 -ln(0.7) = 약 0.356
- B는 미구매 확률인 0.7을 C는 구매확률 0.7을 실제 결과에 부여
- 실제 결과에 부여한 확률이 같아 NLL도 같다.

**Q3.** Total NLL과 Mean NLL은 각각 얼마이며, 평균은 무엇으로 나누나요?

답변: 
- Total NLL은 약 0.936, 반올림 전 합을 고객 수 3으로 나누면 Mean NLL은 약 0.312, 구매한 고객이 2명이라고 2로 나누지 않는다. 미구매 고객을 포함

**Q4.** 전체 로그우도는 얼마이며, Total NLL과 어떤 관계인가요?

답변:
- 전체 로그 우도는 약 -0.936
- Total NLL은 이 값의 부호를 바꾼 약 0.936
- 즉, Total NLL = -전체 로그우도
- Mean NLL은 고객 수 3으로 한 번 더 나눈 값을 의미

---

## 과제. 앞에서 배운 기본 계산 복습

### 문제 3-1. 새로운 데이터의 우도와 MLE 복습하기

#### 문제 설명

교안 연습문제 3의 실제 구매 결과 `y = [1, 1, 0, 1]`과 구매 확률 후보 `[0.5, 0.75, 0.9]`를 사용합니다. 모든 고객의 구매 확률이 공통 p이고, p가 주어졌을 때 구매 여부는 서로 독립이라고 가정합니다.

문제 2-1의 우도 함수와 후보 비교 코드를 그대로 사용하고, 데이터와 후보만 바꾸어 계산하세요.

#### 요구사항

1. 구매 수를 `np.sum(y)`, 미구매 수를 `len(y) - np.sum(y)`로 구해 출력하세요.
2. 다음 우도 함수로 세 후보의 우도를 구하세요. 소수점 아래 여덟 자리까지 표시합니다.
3. `np.argmax()`로 가장 큰 우도의 위치를 찾고 해당 p를 출력하세요. `np.mean(y)`로 MLE도 출력하세요.
4. Q1~Q4에 답하세요.

#### 작성 도움말

```python
def likelihood(p, y):
    observed_prob = (p ** y) * ((1 - p) ** (1 - y))
    return np.prod(observed_prob)

values = np.array([likelihood(p, y) for p in candidates])
best_index = np.argmax(values)
best_p = candidates[best_index]
```

문제 2-1과 같은 순서입니다. 구매가 몇 번, 미구매가 몇 번인지 먼저 세면 L(p)의 거듭제곱 식도 쓸 수 있습니다.

#### 해석 질문

**Q1.** 구매·미구매 수와 이 데이터의 우도 식 L(p)는 무엇인가요?

**Q2.** 세 후보의 우도는 각각 얼마이며, 가장 좋은 후보는 무엇인가요?

**Q3.** 최대 우도의 위치 번호와 관측 평균으로 구한 MLE는 각각 무엇인가요?

**Q4.** p=0.9가 p=0.75보다 큰데도 가장 좋은 후보가 아닌 이유는 무엇인가요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [6]:
# 문제 3-1. 새로운 데이터의 우도와 MLE 복습하기
# 1. 문제에 주어진 배열과 값을 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 계산 결과의 뜻을 적으세요.

y = np.array([1, 1, 0, 1])
candidates = np.array([0.5, 0.75, 0.9])

# 1. 구매 및 미구매 수
purchase_count = np.sum(y)
non_purchase_count = len(y) - np.sum(y) 

print("구매 수:", purchase_count)
print("미구매 수 :", non_purchase_count)

# 2. 우도 함수
def likelihood(p, y):
    observed_prob = (p ** y) * ((1 - p) ** (1 - y))
    return np.prod(observed_prob)

# 후보별 우도 계신
values = np.array([likelihood(p, y) for p in candidates])

for p, value in zip(candidates, values):
    print(f"p={p}: 우도={value:.8f}")

# 3. 최대 우도 후보
best_index = np.argmax(values)
best_p = candidates[best_index]

# MLE
mle = np.mean(y)

print("\n최대 우도 인덱스:", best_index)
print("가장 좋은 후보 p:", best_p)
print("MLE:", mle)


구매 수: 3
미구매 수 : 1
p=0.5: 우도=0.06250000
p=0.75: 우도=0.10546875
p=0.9: 우도=0.07290000

최대 우도 인덱스: 1
가장 좋은 후보 p: 0.75
MLE: 0.75


#### 답변 작성란

**Q1.** 구매·미구매 수와 이 데이터의 우도 식 L(p)는 무엇인가요?

답변: 

- 구매 수는 3번, 미구매 수는 1번이다.
- 우도 식은 L(p) = p³ × (1-p)이다.

**Q2.** 세 후보의 우도는 각각 얼마이며, 가장 좋은 후보는 무엇인가요?

답변: 

- p=0.5일 때 우도는 0.06250000이다.
- p=0.75일 때 우도는 0.10546875이다.
- p=0.9일 때 우도는 0.07290000이다.
- 세 후보 중 우도가 가장 큰 값은 0.10546875이므로 가장 좋은 후보는 p=0.75이다.

**Q3.** 최대 우도의 위치 번호와 관측 평균으로 구한 MLE는 각각 무엇인가요?

답변: 

- 최대 우도의 인덱스 번호는 1(p=0.75)이다.
- 관측평균은 (1+1+0+1) / 4 = 0.75이다.
- 따라서 MLE은 0.75이다.

**Q4.** p=0.9가 p=0.75보다 큰데도 가장 좋은 후보가 아닌 이유는 무엇인가요?

답변:
- p=0.9는 구매할 확률이 높지만, 실제 데이터에는 미구매가 1번 포함되어 있다.

### 문제 3-2. 고객별 확률이 다른 두 모델의 우도 비교하기

#### 문제 설명

교안 연습문제 2의 두 모델 예제입니다. 표의 구매 확률은 이미 계산된 값입니다. 각 모델의 입력 특성과 파라미터가 주어졌을 때 고객들의 구매 여부는 서로 조건부 독립이라고 가정합니다.

| 고객 | 실제 구매 여부 y | 모델 A의 구매 확률 | 모델 B의 구매 확률 |
|---|---:|---:|---:|
| 첫 번째 고객 | 1 | 0.8 | 0.6 |
| 두 번째 고객 | 0 | 0.4 | 0.2 |
| 세 번째 고객 | 1 | 0.6 | 0.5 |

문제 1-2와 같이 실제 결과에 해당하는 확률을 고르고 곱하세요. 이번에는 공통 p 하나 대신 고객별 p 배열이 주어집니다. 배열의 같은 위치가 같은 고객입니다.

#### 요구사항

1. `y = np.array([1,0,1])`, `p_a = np.array([0.8,0.4,0.6])`, `p_b = np.array([0.6,0.2,0.5])`를 만드세요.
2. `(p ** y) * ((1-p) ** (1-y))`를 두 모델에 각각 적용하여 실제 결과의 확률 배열을 출력하세요.
3. `np.prod()`로 두 우도를 계산해 소수점 아래 세 자리로 출력하세요.
4. Q1~Q4에 답하세요.

#### 작성 도움말

두 번째 고객은 미구매입니다. 모델 A에서는 1−0.4를, 모델 B에서는 1−0.2를 사용해야 합니다. 구매 확률 배열을 곧바로 곱하지 않도록 주의하세요.

#### 해석 질문

**Q1.** 각 모델이 실제 결과에 부여한 확률 배열은 무엇인가요?

**Q2.** 두 모델의 우도와 전체 우도 기준으로 선택할 모델은 무엇인가요?

**Q3.** 미구매한 둘째 고객에게 더 높은 관측 확률을 주는 모델은 무엇이며, 전체 선택과 같나요?

**Q4.** 모델 A의 우도 0.288은 모델 A가 참일 확률이 28.8%라는 뜻인가요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [ ]:
# 문제 3-2. 고객별 확률이 다른 두 모델의 우도 비교하기
# 1. 문제에 주어진 배열과 값을 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 계산 결과의 뜻을 적으세요.

# 1
y = np.array([1,0,1])
p_a = np.array([0.8,0.4,0.6])
p_b = np.array([0.6,0.2,0.5])

# 2. 실제 결과에 해당하는 확률 배열
observed_a = (p_a ** y) * ((1 - p_a) ** (1 - y))
observed_b = (p_b ** y) * ((1 - p_b) ** (1 - y))

print("모델 A의 관측 확률 배열:", observed_a)
print("모델 B의 관측 확률 배열:", observed_b)

# 3. 두 모델의 우도 계산
likelihood_a = np.prod(observed_a)
likelihood_b = np.prod(observed_b)

print(f"모델 A의 우도: {likelihood_a:.3f}")
print(f"모델 B의 우도: {likelihood_b:.3f}")

# 우도가 더 큰 모델 선택
if likelihood_a > likelihood_b:
    print("선택된 모델: A")
elif likelihood_b > likelihood_a:
    print("선택된 모델: B")
else:
    print("두 모델의 우도가 같습니다.")


모델 A의 관측 확률 배열: [0.8 0.6 0.6]
모델 B의 관측 확률 배열: [0.6 0.8 0.5]
모델 A의 우도: 0.288
모델 B의 우도: 0.240
선택된 모델: A


#### 답변 작성란

**Q1.** 각 모델이 실제 결과에 부여한 확률 배열은 무엇인가요?

답변: 

- 모델 A가 실제 결과에 부여한 확률 배열은 [0.8, 0.6, 0.6]이다.
- 모델 B가 실제 결과에 부여한 확률 배열은 [0.6, 0.8, 0.5]이다.

**Q2.** 두 모델의 우도와 전체 우도 기준으로 선택할 모델은 무엇인가요?

답변: 

- 모델 A의 우도는 0.8 × 0.6 × 0.6 = 0.288이다.
- 모델 B의 우도는 0.6 × 0.8 × 0.5 = 0.240이다.
- 모델 A의 우도가 모델 B보다 크므로 전체 우도 기준으로 모델 A를 선택한다.

**Q3.** 미구매한 둘째 고객에게 더 높은 관측 확률을 주는 모델은 무엇이며, 전체 선택과 같나요?

답변: 

- 두 번째 고객은 실제로 구매하지 않았다.
- 모델 A가 미구매 결과에 부여한 확률은 1 - 0.4 = 0.6이다.
- 모델 B가 미구매 결과에 부여한 확률은 1 - 0.2 = 0.8이다.
- 따라서 두 번째 고객에게 더 높은 관측 확률을 부여한 모델은 B이다.
- 하지만 전체 우도는 모델 A가 더 높으므로 전체 선택 결과와는 다르다.

**Q4.** 모델 A의 우도 0.288은 모델 A가 참일 확률이 28.8%라는 뜻인가요?

답변:

- 아니요.
- 우도 0.288은 모델 A가 주어졌을 때 실제 관측 데이터 [1, 0, 1]이 나타날 확률을 의미한다.
- 모델 A 자체가 참일 확률이 28.8%라는 뜻은 아니다.
- 우도는 주어진 데이터를 각 모델이 얼마나 잘 설명하는지 비교하는 데 사용한다.
- 따라서 모델 A의 우도가 더 높다는 것은 모델 B보다 관측 데이터를 더 잘 설명한다는 의미이다.

---

## 실습 마무리

다음 질문에 한두 문장으로 답하며 복습하세요.

**1.** 조건부 구매 비율의 분모는 누구인가요?

**2.** 실제 결과가 1 또는 0이면 각각 어떤 확률을 쓰나요?

**3.** Likelihood와 MLE는 어떻게 다른가요?

**4.** 공통 p를 사용하는 독립 베르누이 모델의 MLE는 어떻게 구하나요?

**5.** 전체 로그우도, Total NLL, Mean NLL의 관계는 무엇인가요?

**6.** 같은 데이터에서 우도·로그우도·NLL은 각각 어느 방향을 선택하나요?

### 실습 마무리 답변 작성란

**1.** 답변: 
- "조건"을 만족하는 고객 전체, 장바구니에 추가하겠다는 조건이면 추가한 고객을 분모로 한다.

**2.** 답변: 
- 구매 확률을 p라고 할 때 y = 1에는 p, y=0에는 1-p를 사용
- 단, p는 항상 실제 결과의 확률인 것은 아님

**3.** 답변: 
- Likelihood는 한 파라미터에서 관측 데이터에 부여한 확률을 비교하는 값입니다.
- MLE는 그 우도를 가장 크게 만드는 파라미터를 찾는 방법

**4.** 답변: 
- 구매 수를 전체 고객 수로 나눈다.
- 0, 1 실제값 배열의 평균 np.mean(y)와 같음

**5.** 답변: 
- 관측 확률의 자연로그를 모두 더하면 전체 로그우도, 그 부호를 바꾸면 Total NLL
- Total NLL을 데이터 개수로 나누면 Mean NLL

**6.** 답변:
- 우도와 로그우도는 큰 값, NLL은 작은 값을 선택한다.
- 양수에서 로그는 크기 순서를 유지하고, 마이너스를 붙이면 순서가 뒤집힘.